# Reading OpenSn ASCII cross sections

OpenSn's native cross-section format is a plain-text format intended for readable, version-controlled material data. This tutorial introduces the structure of a non-fissionable file, loads a two-group example, and inspects the resulting `MultiGroupXS` object.

The same `LoadFromOpenSn` call is used for one-group and multigroup files; the `NUM_GROUPS` entry in the file determines the number of groups.

## Minimal file structure

A minimal OpenSn ASCII cross-section file declares its number of energy groups and scattering moments, followed by one or more data blocks:

```text
NUM_GROUPS <number of groups>
NUM_MOMENTS <number of scattering moments>

SIGMA_T_BEGIN
<group> <total cross section>
SIGMA_T_END

SIGMA_A_BEGIN
<group> <absorption cross section>
SIGMA_A_END

TRANSFER_MOMENTS_BEGIN
M_GFROM_GTO_VAL <moment> <departing group> <arrival group> <value>
TRANSFER_MOMENTS_END
```

Group indices and moment indices are zero-based. `NUM_MOMENTS` is the number of moments, so `NUM_MOMENTS 1` supplies only the isotropic (`ell = 0`) scattering moment. `SIGMA_A` and the transfer block are optional; when absorption is omitted, OpenSn can infer it from the total cross section and zeroth transfer matrix. More complete files can also contain group boundaries, velocities, and fission or delayed-neutron data.

## Examine the example file

The tutorial uses `xs_2g.xs`, a two-group file with isotropic scattering. Printing it here keeps the example next to the format description while ensuring that the displayed input is the file that OpenSn will actually read.

In [ ]:
from pathlib import Path

xs_path = Path("xs_2g.xs")
print(xs_path.read_text())

Each `SIGMA_T` row contains a group index followed by its total cross section. Each `M_GFROM_GTO_VAL` row contains the scattering-moment index, departing-group index, arrival-group index, and transfer cross section, in that order.

## Load the file with OpenSn

In [ ]:
import numpy as np
from mpi4py import MPI

from pyopensn.context import Finalize, UseColor
from pyopensn.xs import MultiGroupXS

UseColor(False)

xs_mat = MultiGroupXS()
xs_mat.LoadFromOpenSn(str(xs_path))

## Inspect the imported data

The Python properties provide read-only views of the data used by the solver. They can be converted to lists or NumPy arrays for inspection and post-processing. Because the example omits `SIGMA_A`, the absorption values below are inferred from the total and scattering cross sections.

In [ ]:
print("num_groups       =", xs_mat.num_groups)
print("scattering_order =", xs_mat.scattering_order)
print("sigma_t          =", list(xs_mat.sigma_t))
print("sigma_a          =", np.asarray(xs_mat.sigma_a))

## Finalize the notebook environment

PyOpenSn finalizes automatically in Python script and OpenSn console modes. Jupyter kernels require explicit finalization, and PyOpenSn must be finalized before MPI.

In [ ]:
from IPython import get_ipython


def finalize_env():
    Finalize()
    MPI.Finalize()


ipython_instance = get_ipython()
if ipython_instance is not None:
    ipython_instance.events.register("post_execute", finalize_env)